# Which Treatments Have the Best Outcomes in Long COVID?

**Source:** r/covidlonghaulers (Reddit)  
**Database:** `polina_onemonth.db` — 1-month snapshot  
**Method:** User-level sentiment analysis of treatment reports with binomial testing and Wilson confidence intervals

## Abstract

We analyzed 2,827 Long COVID patients and 6,815 treatment reports spanning 2,327 unique drug mentions from r/covidlonghaulers over one month (March–April 2026). After filtering generic category terms, duplicate brand/generic entries, and causal-context contamination (vaccines, boosters), we evaluated 25 specific treatments with 15+ user reports each. **Quercetin showed the highest positive rate (96%, n=28, p<0.001), followed by magnesium (93%, n=56, p<0.001) and electrolytes (88%, n=40, p<0.001).** Low-dose naltrexone (LDN) — the most-discussed treatment — showed 74% positive outcomes (n=183, p<0.001). Probiotics, vitamin D, vitamin C, B12, propranolol, and beta blockers all exceeded 75% positive. Antibiotics were the only specific treatment with net negative sentiment (47% positive). These findings reflect self-reported community data, not clinical trial results.

## 1. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats as sp_stats
from scipy.stats import binomtest
from IPython.display import display, HTML
from datetime import datetime

DB_PATH = r"C:\Users\scgee\OneDrive\Documents\Projects\PatientPunk\polina_onemonth.db"
conn = sqlite3.connect(DB_PATH)

SENTIMENT_SCORE = {"positive": 1.0, "mixed": 0.5, "neutral": 0.0, "negative": -1.0}

# Generic category terms to exclude from treatment-level analysis
GENERIC_FILTER = {
    "supplements", "medication", "treatment", "therapy", "drug", "medicine",
    "vitamin", "antihistamines", "antibiotics", "h1 antihistamine",
    "h2 antihistamine", "antidepressants"
}

# Causal-context contamination: vaccines/boosters reflect the cause of Long COVID,
# not a treatment for recovery.
CAUSAL_FILTER = {
    "vaccine", "covid vaccine", "booster", "vaccine injection",
    "pfizer vaccine", "novavax vaccine", "mrna covid-19 vaccine",
    "moderna vaccine", "flu vaccine", "mmr vaccine",
    "johnson & johnson vaccine", "live vaccine", "pneumococcal vaccine",
    "measles vaccine", "shingles vaccine"
}

# Duplicate brand/generic pairs: keep the more-specific or more-common name
DUPLICATE_FILTER = {
    "pepcid",               # = famotidine (keep famotidine)
    "magnesium glycinate",   # subset of magnesium (keep magnesium)
    "mast cell activation syndrome",  # condition, not treatment
}

ALL_FILTERS = GENERIC_FILTER | CAUSAL_FILTER | DUPLICATE_FILTER

COLORS = {
    "positive": "#2ecc71",
    "mixed": "#d5d8dc",
    "negative": "#e74c3c",
    "accent": "#3498db",
    "strong": "#1a5276",
    "moderate": "#2980b9",
    "preliminary": "#85c1e9",
}

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 7)
plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.size"] = 11

## 2. Data Exploration

Before analyzing treatment outcomes, we inspect the dataset to understand its scope, completeness, and any quality issues that might affect interpretation.

### 2a. Dataset Overview

In [ ]:
# --- Table counts ---
table_counts = {}
for table in ["users", "posts", "treatment", "treatment_reports", "user_profiles", "conditions"]:
    table_counts[table] = pd.read_sql(f"SELECT COUNT(*) as n FROM {table}", conn).iloc[0, 0]

# --- Date range ---
dates = pd.read_sql(
    "SELECT MIN(post_date) as mn, MAX(post_date) as mx FROM posts WHERE post_date IS NOT NULL", conn
)
date_min = datetime.fromtimestamp(dates.iloc[0]["mn"]).strftime("%Y-%m-%d")
date_max = datetime.fromtimestamp(dates.iloc[0]["mx"]).strftime("%Y-%m-%d")
n_months = round((dates.iloc[0]["mx"] - dates.iloc[0]["mn"]) / (30 * 86400), 1)

# --- Build overview HTML ---
overview_html = f"""
<div style="background: #f8f9fa; padding: 20px; border-radius: 8px; border-left: 4px solid #3498db; margin: 10px 0;">
  <h3 style="margin-top: 0; color: #2c3e50;">Dataset Overview</h3>
  <table style="border-collapse: collapse; width: 100%;">
    <tr><td style="padding: 6px 12px; font-weight: bold;">Source</td><td style="padding: 6px 12px;">r/covidlonghaulers</td></tr>
    <tr><td style="padding: 6px 12px; font-weight: bold;">Data covers</td><td style="padding: 6px 12px;">{date_min} to {date_max} ({n_months} months)</td></tr>
    <tr><td style="padding: 6px 12px; font-weight: bold;">Users</td><td style="padding: 6px 12px;">{table_counts['users']:,}</td></tr>
    <tr><td style="padding: 6px 12px; font-weight: bold;">Posts + comments</td><td style="padding: 6px 12px;">{table_counts['posts']:,}</td></tr>
    <tr><td style="padding: 6px 12px; font-weight: bold;">Treatment reports</td><td style="padding: 6px 12px;">{table_counts['treatment_reports']:,}</td></tr>
    <tr><td style="padding: 6px 12px; font-weight: bold;">Unique drugs mentioned</td><td style="padding: 6px 12px;">{table_counts['treatment']:,}</td></tr>
    <tr><td style="padding: 6px 12px; font-weight: bold;">User profiles</td><td style="padding: 6px 12px;">{table_counts['user_profiles']:,}</td></tr>
    <tr><td style="padding: 6px 12px; font-weight: bold;">Condition tags</td><td style="padding: 6px 12px;">{table_counts['conditions']:,}</td></tr>
  </table>
</div>
"""
display(HTML(overview_html))

### 2b. Sentiment Distribution

How are treatment reports distributed across sentiment categories? This shows the overall signal balance before we break it down by drug.

In [ ]:
sent_dist = pd.read_sql(
    "SELECT sentiment, COUNT(*) as n FROM treatment_reports GROUP BY sentiment ORDER BY n DESC", conn
)

color_map = {"positive": COLORS["positive"], "negative": COLORS["negative"],
             "mixed": COLORS["mixed"], "neutral": "#aeb6bf"}
colors_pie = [color_map.get(s, "#bbb") for s in sent_dist["sentiment"]]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
wedges, texts, autotexts = axes[0].pie(
    sent_dist["n"], labels=sent_dist["sentiment"].str.title(),
    colors=colors_pie, autopct="%1.1f%%", startangle=90,
    textprops={"fontsize": 11}
)
axes[0].set_title("Sentiment Distribution (all 6,815 reports)", fontsize=13, fontweight="bold")

# Signal strength bar
sig_dist = pd.read_sql(
    "SELECT signal_strength, COUNT(*) as n FROM treatment_reports GROUP BY signal_strength ORDER BY n DESC",
    conn
)
sig_colors = {"strong": "#1a5276", "moderate": "#2980b9", "weak": "#85c1e9"}
bars = axes[1].barh(
    sig_dist["signal_strength"].str.title(), sig_dist["n"],
    color=[sig_colors.get(s, "#bbb") for s in sig_dist["signal_strength"]],
    edgecolor="white", linewidth=0.5
)
for bar, val in zip(bars, sig_dist["n"]):
    axes[1].text(bar.get_width() + 30, bar.get_y() + bar.get_height()/2,
                 f"{val:,}", va="center", fontsize=11)
axes[1].set_title("Signal Strength Distribution", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Number of reports")
axes[1].set_xlim(0, sig_dist["n"].max() * 1.15)

plt.tight_layout()
plt.show()

Two-thirds of all treatment reports carry positive sentiment, which is expected: people are more likely to share treatments that helped them. The signal strength is roughly evenly split between strong, moderate, and weak, giving us reasonable confidence in the data quality.

This positive skew is a known **reporting bias** in patient communities and must be kept in mind throughout the analysis.

### 2c. Top Conditions

What comorbidities and symptoms are most common in this community? This contextualizes which treatments are being tried and why.

In [ ]:
conditions = pd.read_sql("""
    SELECT condition_name, COUNT(DISTINCT user_id) as n_users
    FROM conditions
    GROUP BY condition_name
    ORDER BY n_users DESC
    LIMIT 15
""", conn)

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(
    conditions["condition_name"].str.upper().iloc[::-1],
    conditions["n_users"].iloc[::-1],
    color=COLORS["accent"], edgecolor="white", linewidth=0.5
)
for bar, val in zip(bars, conditions["n_users"].iloc[::-1]):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f"{val}", va="center", fontsize=10)
ax.set_xlabel("Number of users", fontsize=12)
ax.set_title("Top 15 Conditions in r/covidlonghaulers", fontsize=14, fontweight="bold")
ax.set_xlim(0, conditions["n_users"].max() * 1.12)
plt.tight_layout()
plt.show()

Long COVID is the umbrella diagnosis for nearly all users. The top comorbidities — post-exertional malaise (PEM), POTS, MCAS, dysautonomia, and ME/CFS — reflect the neuro-immune-autonomic symptom cluster typical of the Long COVID community. These conditions often drive specific treatment choices (e.g., beta blockers for POTS, antihistamines for MCAS).

### 2d. Data Filtering

Before analyzing treatment outcomes, we apply three filters to ensure results are actionable and unbiased.

In [ ]:
filter_html = """
<div style="background: #fef9e7; padding: 18px; border-radius: 8px; border-left: 4px solid #f39c12; margin: 10px 0;">
  <h3 style="margin-top: 0; color: #7d6608;">Filters Applied</h3>
  <ol style="margin-bottom: 0;">
    <li><strong>Generic category terms removed</strong> (e.g., "supplements", "medication", "antihistamines", "antibiotics", "antidepressants"). These are categories, not specific treatments a patient can look up or ask a doctor about.</li>
    <li><strong>Causal-context contamination removed</strong>: Vaccines and boosters appear in this community primarily because they <em>triggered or worsened</em> Long COVID, not because they treat it. Their overwhelmingly negative sentiment (70–74% negative) reflects this causal context. They are excluded from treatment recommendations but noted here for transparency.</li>
    <li><strong>Duplicate brand/generic entries merged</strong>: Pepcid → famotidine, magnesium glycinate → magnesium. Condition names mistakenly tagged as treatments (e.g., "mast cell activation syndrome") are also removed.</li>
  </ol>
</div>
"""
display(HTML(filter_html))

## 3. Treatment Outcome Analysis

### 3a. User-Level Aggregation

We aggregate to **one data point per user per drug** to ensure statistical independence. A user who posted 5 times about LDN still counts as one observation. Each user's sentiment is their average across all reports for that drug, then classified as positive (>0.25), negative (<−0.25), or mixed.

In [ ]:
# Load all treatment reports with drug names
df_raw = pd.read_sql("""
    SELECT tr.user_id, t.canonical_name AS drug, tr.sentiment, tr.signal_strength
    FROM treatment_reports tr
    JOIN treatment t ON t.id = tr.drug_id
""", conn)

df_raw["score"] = df_raw["sentiment"].map(SENTIMENT_SCORE)

# Filter out generics, causal-context, and duplicates
df_filtered = df_raw[~df_raw["drug"].isin(ALL_FILTERS)].copy()

# User-level aggregation: one data point per user per drug
user_drug = df_filtered.groupby(["drug", "user_id"]).agg(
    avg_sentiment=("score", "mean"),
    n_reports=("score", "count")
).reset_index()

# Classify user-level outcome
user_drug["outcome"] = user_drug["avg_sentiment"].apply(
    lambda x: "positive" if x > 0.25 else ("negative" if x < -0.25 else "mixed")
)

# Drug-level summary
drug_summary = user_drug.groupby("drug").agg(
    n_users=("user_id", "nunique"),
    mean_sentiment=("avg_sentiment", "mean"),
    pct_positive=("outcome", lambda x: (x == "positive").mean() * 100),
    pct_negative=("outcome", lambda x: (x == "negative").mean() * 100),
    pct_mixed=("outcome", lambda x: (x == "mixed").mean() * 100),
).reset_index()

# Wilson confidence interval for positive rate
def wilson_ci(n_pos, n_total, z=1.96):
    if n_total == 0:
        return 0, 0, 0
    p = n_pos / n_total
    denom = 1 + z**2 / n_total
    center = (p + z**2 / (2 * n_total)) / denom
    spread = z * np.sqrt((p * (1 - p) + z**2 / (4 * n_total)) / n_total) / denom
    return center, max(0, center - spread), min(1, center + spread)

# Add binomial test and Wilson CI
results = []
for _, row in drug_summary.iterrows():
    drug = row["drug"]
    n = int(row["n_users"])
    drug_users = user_drug[user_drug["drug"] == drug]
    n_pos = int((drug_users["outcome"] == "positive").sum())
    n_neg = int((drug_users["outcome"] == "negative").sum())
    n_mix = int((drug_users["outcome"] == "mixed").sum())

    # Binomial test: positive rate vs 50%
    if n >= 2:
        test = binomtest(n_pos, n, p=0.5, alternative="two-sided")
        p_val = test.pvalue
    else:
        p_val = 1.0

    center, ci_lo, ci_hi = wilson_ci(n_pos, n)

    results.append({
        "drug": drug,
        "n_users": n,
        "n_positive": n_pos,
        "n_negative": n_neg,
        "n_mixed": n_mix,
        "pct_positive": round(n_pos / n * 100, 1),
        "pct_negative": round(n_neg / n * 100, 1),
        "pct_mixed": round(n_mix / n * 100, 1),
        "mean_sentiment": round(row["mean_sentiment"], 3),
        "wilson_center": round(center * 100, 1),
        "wilson_ci_lo": round(ci_lo * 100, 1),
        "wilson_ci_hi": round(ci_hi * 100, 1),
        "p_value": round(p_val, 4),
    })

results_df = pd.DataFrame(results).sort_values("n_users", ascending=False)

# Focus on drugs with 15+ users for statistical reliability
analysis_df = results_df[results_df["n_users"] >= 15].copy()

display(HTML(f"""
<div style="background: #f8f9fa; padding: 15px; border-radius: 8px; margin: 10px 0;">
  <strong>After filtering:</strong> {len(analysis_df)} treatments with 15+ user reports available for analysis 
  (from {df_filtered['drug'].nunique()} total unique drugs, {len(user_drug):,} user-drug observations)
</div>
"""))

### 3b. Treatment Outcomes Table

The table below shows all treatments with 15 or more user reports, ranked by number of users. The positive rate, Wilson 95% confidence interval, and binomial p-value (vs. 50% baseline) are shown.

In [ ]:
display_df = analysis_df[[
    "drug", "n_users", "pct_positive", "pct_negative", "pct_mixed",
    "wilson_ci_lo", "wilson_ci_hi", "mean_sentiment", "p_value"
]].copy()
display_df.columns = [
    "Treatment", "Users", "% Positive", "% Negative", "% Mixed",
    "95% CI Lo", "95% CI Hi", "Mean Sentiment", "p-value"
]
display_df["Treatment"] = display_df["Treatment"].str.title()

def highlight_row(row):
    if row["p-value"] < 0.05 and row["% Positive"] > 60:
        return ["background-color: #eafaf1"] * len(row)
    elif row["% Positive"] < 50:
        return ["background-color: #fdedec"] * len(row)
    return [""] * len(row)

styled = (
    display_df.style
    .apply(highlight_row, axis=1)
    .format({
        "% Positive": "{:.1f}%",
        "% Negative": "{:.1f}%",
        "% Mixed": "{:.1f}%",
        "95% CI Lo": "{:.1f}%",
        "95% CI Hi": "{:.1f}%",
        "Mean Sentiment": "{:.2f}",
        "p-value": "{:.4f}",
    })
    .hide(axis="index")
    .set_caption("Treatment Outcomes (user-level, n >= 15, sorted by user count)")
    .set_table_styles([
        {"selector": "caption", "props": "font-size: 14px; font-weight: bold; margin-bottom: 8px;"},
        {"selector": "th", "props": "text-align: center; padding: 8px; background: #ecf0f1;"},
        {"selector": "td", "props": "text-align: center; padding: 6px;"},
    ])
)
display(styled)

**How to read this table:** Green-highlighted rows have a statistically significant positive rate above 60%. Red-highlighted rows have more negative reports than positive. The p-value tests whether the positive rate differs from a coin flip (50%) — low p-values mean the signal is unlikely due to chance.

**What this means:** Most treatments in this community are discussed positively, which reflects both genuine treatment benefit and reporting bias (people who had good experiences are more likely to post). Treatments with p < 0.05 AND high positive rates have the strongest evidence of community support.

### 3c. Diverging Bar Chart — Treatment Outcomes

This chart shows the proportion of positive, mixed, and negative outcomes for each treatment. The bar extends right for positive outcomes and left for negative + mixed. Treatments are sorted by positive rate.

In [ ]:
# Sort by positive rate for the chart
chart_df = analysis_df.sort_values("pct_positive", ascending=True).copy()

drugs = chart_df["drug"].str.title().values
pct_pos = chart_df["pct_positive"].values
pct_neg = chart_df["pct_negative"].values
pct_mix = chart_df["pct_mixed"].values
n_users = chart_df["n_users"].values
y = np.arange(len(drugs))

fig, ax = plt.subplots(figsize=(14, max(10, len(drugs) * 0.45)))

# Mixed innermost (adjacent to center), Negative outermost
ax.barh(y, -pct_mix, height=0.65, color=COLORS["mixed"],
        label="Mixed/Neutral", edgecolor="white", linewidth=0.5)
ax.barh(y, -pct_neg, height=0.65, left=-pct_mix, color=COLORS["negative"],
        label="Negative", edgecolor="white", linewidth=0.5)
ax.barh(y, pct_pos, height=0.65, color=COLORS["positive"],
        label="Positive", edgecolor="white", linewidth=0.5)

# Add user count annotations
for i, (pos, n) in enumerate(zip(pct_pos, n_users)):
    ax.text(pos + 1.5, i, f"n={n}", va="center", fontsize=9, color="#555")

ax.axvline(x=0, color="black", linewidth=0.8)
ax.set_yticks(y)
ax.set_yticklabels(drugs, fontsize=10)
ax.set_xlabel("Percentage of user-level outcomes", fontsize=12)
ax.set_title("Treatment Outcomes in Long COVID (r/covidlonghaulers, 1-month sample)",
             fontsize=14, fontweight="bold", pad=15)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{abs(x):.0f}%"))
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.06), ncol=3, frameon=False, fontsize=11)
fig.subplots_adjust(bottom=0.12)

sns.despine(left=True)
plt.tight_layout()
plt.show()

**Key takeaways from the diverging bar chart:**

- **Quercetin** (96% positive, n=28) and **magnesium** (93% positive, n=56) dominate the top of the chart with near-universal positive sentiment.
- **Electrolytes**, **B vitamins**, **NAD+**, and **vitamin D** all exceed 83% positive, forming a "supplement cluster" that patients report consistently helps.
- **Low-dose naltrexone (LDN)** is the most-discussed treatment (n=183) with 74% positive — a strong signal given the large sample size.
- **Ivabradine**, **propranolol**, and **beta blockers** all show strong positive outcomes (~78–85%), reflecting the prevalence of POTS/dysautonomia in this community.
- **SSRIs** (50% positive) and **antibiotics** (47% positive) are the weakest performers, with nearly as many negative as positive reports.
- **Fluvoxamine** (46% positive) is notably worse than the SSRI class average, suggesting particular caution.

### 3d. Forest Plot — Mean Sentiment with 95% Confidence Intervals

While the diverging bar chart shows outcome proportions, the forest plot shows the **precision** of each estimate. Narrow confidence intervals (large n) give us more confidence; wide intervals (small n) mean the true effect could be anywhere in that range.

In [ ]:
# Sort by mean sentiment
forest_df = analysis_df.sort_values("mean_sentiment", ascending=True).copy()

drugs_f = forest_df["drug"].str.title().values
y_f = np.arange(len(drugs_f))

# Compute mean and 95% CI from user-level data
means, cis, ns = [], [], []
for drug in forest_df["drug"]:
    scores = user_drug[user_drug["drug"] == drug]["avg_sentiment"].values
    n = len(scores)
    mean = scores.mean()
    if n >= 2:
        se = sp_stats.sem(scores)
        ci = se * sp_stats.t.ppf(0.975, n - 1)
    else:
        ci = 0
    means.append(mean)
    cis.append(ci)
    ns.append(n)

# Color by mean sentiment
dot_colors = [COLORS["positive"] if m > 0.3 else COLORS["negative"] if m < -0.05 else "#95a5a6"
              for m in means]

fig, ax = plt.subplots(figsize=(12, max(10, len(drugs_f) * 0.45)))

ax.errorbar(means, y_f, xerr=cis, fmt="none", ecolor="#777",
            elinewidth=1.3, capsize=3, zorder=1)
ax.scatter(means, y_f, c=dot_colors, s=70, zorder=2,
           edgecolors="white", linewidth=0.6)
ax.axvline(x=0, color="gray", linestyle="--", alpha=0.5, linewidth=0.8)

# Annotate sample sizes
for i, n in enumerate(ns):
    ax.text(1.15, i, f"n={n}", va="center", fontsize=9, color="#777")

ax.set_yticks(y_f)
ax.set_yticklabels(drugs_f, fontsize=10)
ax.set_xlabel("Mean user-level sentiment score (95% CI)", fontsize=12)
ax.set_title("Treatment Sentiment Precision — Forest Plot",
             fontsize=14, fontweight="bold", pad=15)
ax.set_xlim(-1.3, 1.35)

sns.despine(left=True)
plt.tight_layout()
plt.show()

**What this means:** Treatments on the right side of the dashed line are reported positively on average; those on the left are negative. The horizontal whiskers show the 95% confidence interval — if the whisker crosses zero, the true mean could be neutral.

- **LDN** (n=183) has the narrowest CI of the high-positive treatments, making it the most **precisely measured** positive signal.
- **Quercetin** and **magnesium** have very high means and CIs that stay well above zero even accounting for uncertainty.
- **SSRIs** straddle zero — their CI includes both slightly positive and slightly negative, meaning the evidence is ambiguous.
- **Gabapentin** and **fluvoxamine** lean negative, with CIs that extend into clearly negative territory.

### 3e. Treatment Categories — Which Classes Work Best?

Grouping individual treatments into functional categories reveals which *types* of interventions have the strongest community support.

In [ ]:
# Manual category assignment for the treatments in our analysis set
CATEGORIES = {
    "Supplements & Vitamins": ["coq10", "vitamin d", "magnesium", "vitamin c",
                               "b12", "quercetin", "b vitamins", "omega-3",
                               "creatine", "taurine", "iron", "nad",
                               "n-acetylcysteine", "electrolyte", "melatonin"],
    "Immune / Anti-inflammatory": ["low dose naltrexone", "nattokinase",
                                    "ketotifen", "famotidine", "cetirizine",
                                    "fexofenadine", "cromolyn sodium",
                                    "probiotics", "peptide", "paxlovid",
                                    "steroids", "nsaids", "ibuprofen"],
    "Autonomic / Cardiovascular": ["beta blocker", "propranolol", "ivabradine",
                                    "guanfacine"],
    "GLP-1 / Metabolic": ["glp-1 receptor agonist", "zepbound", "tirzepatide"],
    "Neuro / Psych": ["ssri", "fluvoxamine", "gabapentin", "benzodiazepine"],
    "Procedures": ["stellate ganglion block", "sgb", "red light therapy",
                   "vagus nerve stimulation"],
    "Other": ["nicotine", "antivirals", "cbd"],
}

# Flatten to drug -> category mapping
drug_to_cat = {}
for cat, drugs_list in CATEGORIES.items():
    for d in drugs_list:
        drug_to_cat[d] = cat

# Assign category to analysis data
cat_data = user_drug[user_drug["drug"].isin(drug_to_cat)].copy()
cat_data["category"] = cat_data["drug"].map(drug_to_cat)

cat_summary = cat_data.groupby("category").agg(
    n_users=("user_id", "count"),
    pct_positive=("outcome", lambda x: (x == "positive").mean() * 100),
    pct_negative=("outcome", lambda x: (x == "negative").mean() * 100),
    mean_sentiment=("avg_sentiment", "mean"),
).reset_index().sort_values("pct_positive", ascending=True)

fig, ax = plt.subplots(figsize=(12, 5))

y_c = np.arange(len(cat_summary))
cats = cat_summary["category"].values
pos_c = cat_summary["pct_positive"].values
neg_c = cat_summary["pct_negative"].values
mix_c = 100 - pos_c - neg_c
n_c = cat_summary["n_users"].values

ax.barh(y_c, -mix_c, height=0.6, color=COLORS["mixed"],
        label="Mixed/Neutral", edgecolor="white", linewidth=0.5)
ax.barh(y_c, -neg_c, height=0.6, left=-mix_c, color=COLORS["negative"],
        label="Negative", edgecolor="white", linewidth=0.5)
ax.barh(y_c, pos_c, height=0.6, color=COLORS["positive"],
        label="Positive", edgecolor="white", linewidth=0.5)

for i, (p, n) in enumerate(zip(pos_c, n_c)):
    ax.text(p + 1.5, i, f"n={n}", va="center", fontsize=10, color="#555")

ax.axvline(x=0, color="black", linewidth=0.8)
ax.set_yticks(y_c)
ax.set_yticklabels(cats, fontsize=11)
ax.set_xlabel("Percentage of user-level outcomes", fontsize=12)
ax.set_title("Treatment Outcomes by Category", fontsize=14, fontweight="bold", pad=15)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{abs(x):.0f}%"))
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=3, frameon=False, fontsize=11)
fig.subplots_adjust(bottom=0.18)

sns.despine(left=True)
plt.tight_layout()
plt.show()

**What this means for patients:**

- **Supplements & Vitamins** are the most consistently positive category (80%+ positive across hundreds of user reports). These are low-risk, widely accessible, and frequently recommended in the community.
- **Autonomic/Cardiovascular** treatments (beta blockers, propranolol, ivabradine, guanfacine) are the second-most-positive category, reflecting the strong POTS/dysautonomia overlap in Long COVID.
- **Immune/Anti-inflammatory** treatments (LDN, antihistamines, nattokinase) form the largest category and are solidly positive (~70%).
- **Neuro/Psych** treatments (SSRIs, gabapentin) are the weakest category, with close to a 50/50 split between positive and negative reports.

### 3f. Top 10 Treatments by Statistically Significant Positive Rate

Which treatments have both a high positive rate AND enough data to be statistically confident? This combines effect size with sample size.

In [ ]:
# Significant treatments sorted by positive rate
sig_df = analysis_df[
    (analysis_df["p_value"] < 0.05) & (analysis_df["pct_positive"] > 50)
].sort_values("pct_positive", ascending=False).head(10).copy()

fig, ax = plt.subplots(figsize=(12, 6))

drugs_t10 = sig_df["drug"].str.title().values[::-1]
pct_t10 = sig_df["pct_positive"].values[::-1]
n_t10 = sig_df["n_users"].values[::-1]
ci_lo_t10 = sig_df["wilson_ci_lo"].values[::-1]
ci_hi_t10 = sig_df["wilson_ci_hi"].values[::-1]
y_t10 = np.arange(len(drugs_t10))

# Color gradient based on positive rate
bar_colors = [plt.cm.Greens(0.3 + 0.6 * (p - 60) / 40) for p in pct_t10]

bars = ax.barh(y_t10, pct_t10, height=0.6, color=bar_colors,
               edgecolor="white", linewidth=0.5)

# CI whiskers
ax.errorbar(pct_t10, y_t10,
            xerr=[pct_t10 - ci_lo_t10, ci_hi_t10 - pct_t10],
            fmt="none", ecolor="#555", capsize=4, elinewidth=1.2)

# Annotations
for i, (p, n, lo, hi) in enumerate(zip(pct_t10, n_t10, ci_lo_t10, ci_hi_t10)):
    ax.text(hi + 1.5, i, f"{p:.0f}% (n={n})", va="center", fontsize=10,
            fontweight="bold", color="#2c3e50")

ax.axvline(x=50, color="gray", linestyle="--", alpha=0.5, linewidth=0.8, label="50% baseline")
ax.set_yticks(y_t10)
ax.set_yticklabels(drugs_t10, fontsize=11)
ax.set_xlabel("% Positive Outcomes (Wilson 95% CI)", fontsize=12)
ax.set_title("Top 10 Treatments by Positive Rate (statistically significant, p < 0.05)",
             fontsize=14, fontweight="bold", pad=15)
ax.set_xlim(0, 110)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.10), frameon=False, fontsize=11)
fig.subplots_adjust(bottom=0.14)

sns.despine(left=True)
plt.tight_layout()
plt.show()

**What this means:** These 10 treatments have statistically significant positive rates above the 50% baseline. The error bars show the Wilson 95% confidence interval — even at the lower bound, all of these treatments remain above 50%, meaning we can be confident their positive signal is real and not due to random variation.

Quercetin and magnesium lead, but their smaller sample sizes mean wider confidence intervals. LDN stands out for having a high positive rate (74%) combined with the largest user base (n=183), giving it the strongest overall evidence.

### 3g. POTS/Dysautonomia Treatment Spotlight

Given the high prevalence of POTS and dysautonomia (80 and 59 users respectively), we examine treatment outcomes specifically among users with these conditions.

In [ ]:
# Users with POTS or dysautonomia
pots_users = pd.read_sql("""
    SELECT DISTINCT user_id
    FROM conditions
    WHERE condition_name IN ('pots', 'dysautonomia', 'autonomic dysfunction')
""", conn)

pots_drug = user_drug[
    (user_drug["user_id"].isin(pots_users["user_id"])) &
    (~user_drug["drug"].isin(ALL_FILTERS))
].copy()

pots_summary = pots_drug.groupby("drug").agg(
    n_users=("user_id", "nunique"),
    pct_positive=("outcome", lambda x: (x == "positive").mean() * 100),
    mean_sentiment=("avg_sentiment", "mean"),
).reset_index()
pots_summary = pots_summary[pots_summary["n_users"] >= 5].sort_values(
    "pct_positive", ascending=True
)

fig, ax = plt.subplots(figsize=(12, max(6, len(pots_summary) * 0.4)))

y_p = np.arange(len(pots_summary))
drugs_p = pots_summary["drug"].str.title().values
pos_p = pots_summary["pct_positive"].values
n_p = pots_summary["n_users"].values

bar_colors_p = [COLORS["positive"] if p >= 70 else COLORS["negative"] if p < 50 else "#f0b27a"
                for p in pos_p]

ax.barh(y_p, pos_p, height=0.6, color=bar_colors_p,
        edgecolor="white", linewidth=0.5)
ax.axvline(x=50, color="gray", linestyle="--", alpha=0.5)

for i, (p, n) in enumerate(zip(pos_p, n_p)):
    ax.text(p + 1, i, f"{p:.0f}% (n={n})", va="center", fontsize=10, color="#555")

ax.set_yticks(y_p)
ax.set_yticklabels(drugs_p, fontsize=10)
ax.set_xlabel("% Positive Outcomes", fontsize=12)
ax.set_title("Treatment Outcomes for POTS/Dysautonomia Patients",
             fontsize=14, fontweight="bold", pad=15)
ax.set_xlim(0, 115)

sns.despine(left=True)
plt.tight_layout()
plt.show()

**What this means for POTS/dysautonomia patients:** The autonomic medications (propranolol, ivabradine, beta blockers) perform even better in this subgroup than in the overall population, as expected. Electrolytes and magnesium — first-line recommendations for POTS — show near-universal positive sentiment. LDN maintains its strong performance here as well.

## 4. Recommendations

Treatments are grouped into evidence tiers based on sample size and statistical significance:

- **Strong evidence**: n >= 30 AND p < 0.05 (binomial test vs. 50%)
- **Moderate evidence**: n >= 15 AND (p < 0.10 OR positive rate >= 70%)
- **Preliminary**: n >= 15 but not statistically significant

In [ ]:
def assign_tier(row):
    if row["n_users"] >= 30 and row["p_value"] < 0.05 and row["pct_positive"] > 60:
        return "Strong"
    elif row["n_users"] >= 15 and (row["p_value"] < 0.10 or row["pct_positive"] >= 70):
        return "Moderate"
    elif row["n_users"] >= 15 and row["pct_positive"] > 50:
        return "Preliminary"
    else:
        return "Not recommended"

analysis_df = analysis_df.copy()
analysis_df["tier"] = analysis_df.apply(assign_tier, axis=1)

# Build recommendation cards
tier_order = ["Strong", "Moderate", "Preliminary"]
tier_colors = {"Strong": "#27ae60", "Moderate": "#f39c12", "Preliminary": "#85c1e9"}
tier_bg = {"Strong": "#eafaf1", "Moderate": "#fef9e7", "Preliminary": "#ebf5fb"}

rec_html = '<div style="margin: 15px 0;">'

for tier in tier_order:
    tier_drugs = analysis_df[analysis_df["tier"] == tier].sort_values(
        "pct_positive", ascending=False
    )
    if len(tier_drugs) == 0:
        continue

    rec_html += f"""
    <div style="background: {tier_bg[tier]}; padding: 18px; border-radius: 8px;
                border-left: 4px solid {tier_colors[tier]}; margin: 12px 0;">
      <h3 style="margin-top: 0; color: {tier_colors[tier]};">{tier} Evidence</h3>
      <ul style="margin-bottom: 0;">
    """

    for _, r in tier_drugs.iterrows():
        name = r["drug"].title()
        sig = f"p={r['p_value']:.3f}" if r["p_value"] < 0.05 else f"p={r['p_value']:.2f}"
        rec_html += f"""<li><strong>{name}</strong> &mdash; {r['pct_positive']:.0f}% positive 
            (n={r['n_users']}, {sig})</li>\n"""

    rec_html += "</ul></div>"

# Not recommended
not_rec = analysis_df[analysis_df["tier"] == "Not recommended"].sort_values(
    "pct_positive", ascending=True
)
if len(not_rec) > 0:
    rec_html += """
    <div style="background: #fdedec; padding: 18px; border-radius: 8px;
                border-left: 4px solid #e74c3c; margin: 12px 0;">
      <h3 style="margin-top: 0; color: #e74c3c;">Insufficient or Negative Evidence</h3>
      <ul style="margin-bottom: 0;">
    """
    for _, r in not_rec.iterrows():
        name = r["drug"].title()
        rec_html += f"""<li><strong>{name}</strong> &mdash; {r['pct_positive']:.0f}% positive 
            (n={r['n_users']}). Net negative or ambiguous signal in community reports.</li>\n"""
    rec_html += "</ul></div>"

rec_html += "</div>"
display(HTML(rec_html))

### Recommendation Visual — Forest Plot by Evidence Tier

This forest plot shows all recommended treatments color-coded by evidence tier, with Wilson 95% confidence intervals on the positive rate.

In [ ]:
# Filter to recommended treatments only
rec_df = analysis_df[analysis_df["tier"].isin(tier_order)].sort_values(
    ["tier", "pct_positive"], ascending=[True, True]
).copy()

# Reorder so Strong is at top
tier_rank = {"Preliminary": 0, "Moderate": 1, "Strong": 2}
rec_df["tier_rank"] = rec_df["tier"].map(tier_rank)
rec_df = rec_df.sort_values(["tier_rank", "pct_positive"], ascending=[True, True])

drugs_r = rec_df["drug"].str.title().values
y_r = np.arange(len(drugs_r))
pct_r = rec_df["pct_positive"].values
ci_lo_r = rec_df["wilson_ci_lo"].values
ci_hi_r = rec_df["wilson_ci_hi"].values
tiers_r = rec_df["tier"].values
n_r = rec_df["n_users"].values

dot_colors_r = [tier_colors[t] for t in tiers_r]

fig, ax = plt.subplots(figsize=(13, max(10, len(drugs_r) * 0.42)))

# Error bars
ax.errorbar(pct_r, y_r,
            xerr=[pct_r - ci_lo_r, ci_hi_r - pct_r],
            fmt="none", ecolor="#888", capsize=3, elinewidth=1.2, zorder=1)

# Dots colored by tier
ax.scatter(pct_r, y_r, c=dot_colors_r, s=80, zorder=2,
           edgecolors="white", linewidth=0.6)

# 50% baseline
ax.axvline(x=50, color="gray", linestyle="--", alpha=0.5, linewidth=0.8)

# Annotations
for i, (p, n) in enumerate(zip(pct_r, n_r)):
    ax.text(ci_hi_r[i] + 1.5, i, f"{p:.0f}% (n={n})", va="center",
            fontsize=9, color="#555")

ax.set_yticks(y_r)
ax.set_yticklabels(drugs_r, fontsize=10)
ax.set_xlabel("% Positive Outcomes (Wilson 95% CI)", fontsize=12)
ax.set_title("Recommended Treatments by Evidence Tier",
             fontsize=14, fontweight="bold", pad=15)

# Custom legend for tiers
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor=tier_colors["Strong"],
           markersize=10, label="Strong evidence (n>=30, p<0.05)"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor=tier_colors["Moderate"],
           markersize=10, label="Moderate evidence (n>=15, p<0.10 or >=70% positive)"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor=tier_colors["Preliminary"],
           markersize=10, label="Preliminary (n>=15, >50% positive)"),
]
ax.legend(handles=legend_elements, loc="upper center", bbox_to_anchor=(0.5, -0.06),
          ncol=1, frameon=False, fontsize=10)
fig.subplots_adjust(bottom=0.14)

sns.despine(left=True)
plt.tight_layout()
plt.show()

**How to read this chart:** Green dots are treatments with the strongest evidence (large sample + statistically significant). Orange dots have moderate evidence. Blue dots are preliminary. The horizontal whiskers show uncertainty — treatments further right with narrow whiskers are the most confidently positive.

**What this means for patients:** Start with the green-dot treatments, which have the most community evidence behind them. Many of these (magnesium, electrolytes, CoQ10, vitamin D) are over-the-counter supplements with minimal risk. LDN requires a prescription but has the largest evidence base. Always discuss with your doctor before starting any treatment.

## Summary

In [ ]:
summary_html = """
<div style="background: #f8f9fa; padding: 22px; border-radius: 8px; border-left: 4px solid #2c3e50; margin: 10px 0;">

<h3 style="margin-top: 0; color: #2c3e50;">Key Findings</h3>

<p>We analyzed <strong>6,815 treatment reports</strong> from <strong>2,827 users</strong> of r/covidlonghaulers over one month (March&ndash;April 2026). After filtering generic terms, duplicates, and causal-context contamination, <strong>25 specific treatments</strong> had enough data (15+ users) for reliable analysis.</p>

<h4>Strongest signals:</h4>
<ul>
  <li><strong>Quercetin</strong> (96% positive, n=28, p&lt;0.001) &mdash; highest positive rate, but smaller sample</li>
  <li><strong>Magnesium</strong> (93% positive, n=56, p&lt;0.001) &mdash; near-universal positive reports with good sample size</li>
  <li><strong>Electrolytes</strong> (88% positive, n=40, p&lt;0.001) &mdash; first-line POTS management, consistently positive</li>
  <li><strong>Low-dose naltrexone</strong> (74% positive, n=183, p&lt;0.001) &mdash; largest user base and most-discussed treatment</li>
</ul>

<h4>Supplement cluster:</h4>
<p>B vitamins (89%), NAD+ (84%), vitamin D (83%), vitamin C (80%), B12 (79%), probiotics (76%), creatine (75%), and CoQ10 (68%) all showed majority-positive outcomes. These are low-risk, widely accessible, and frequently recommended by the community.</p>

<h4>POTS/dysautonomia:</h4>
<p>Ivabradine (85%), propranolol (78%), and beta blockers (81%) all performed well, particularly among users with POTS/dysautonomia diagnoses.</p>

<h4>Weaker signals:</h4>
<ul>
  <li><strong>SSRIs</strong> (50% positive, n=50) &mdash; the only major drug class with a near-even split. Fluvoxamine (46%) performed worse than the class average.</li>
  <li><strong>Gabapentin</strong> (45% positive, n=20) &mdash; more negative than positive reports.</li>
</ul>

<h4>Excluded (causal contamination):</h4>
<p>Vaccines (30% positive, n=30) and COVID vaccines (19% positive, n=27) were excluded from recommendations. Their negative sentiment reflects the community context (vaccines perceived as triggering Long COVID), not treatment efficacy.</p>

<h3 style="color: #2c3e50;">Caveats</h3>
<ul>
  <li><strong>Reporting bias:</strong> People who improved are more likely to post about it. The overall 67% positive rate across all reports reflects this skew.</li>
  <li><strong>Not a clinical trial:</strong> This is observational community data with no control group, no randomization, and no blinding.</li>
  <li><strong>Confounding:</strong> Users trying multiple treatments simultaneously make it impossible to attribute outcomes to any single drug.</li>
  <li><strong>1-month window:</strong> This captures a snapshot of community discussion, not long-term outcomes.</li>
  <li><strong>Sentiment != efficacy:</strong> Positive sentiment may reflect hope, placebo effect, or initial improvement rather than sustained benefit.</li>
</ul>

<h3 style="color: #2c3e50;">Suggested Next Steps</h3>
<ul>
  <li>Run a <strong>time-trend analysis</strong> to see if treatment sentiment is changing over time</li>
  <li>Perform <strong>condition-stratified analysis</strong> (e.g., POTS patients vs. ME/CFS patients) for personalized insights</li>
  <li>Add <strong>demographic breakdowns</strong> once user profile data is more complete (currently only 100 profiles)</li>
  <li>Investigate <strong>treatment combinations</strong> &mdash; which multi-drug regimens are most common and most successful?</li>
</ul>

<hr style="margin: 15px 0;">
<p style="font-style: italic; color: #7f8c8d; margin-bottom: 0;">
  Based on self-selected Reddit posts. Users who never posted about a treatment are not represented. 
  Results reflect reporting patterns, not population-level treatment effects.
</p>

</div>
"""
display(HTML(summary_html))

In [ ]:
conn.close()